# 03. Ingeniería de Características para el Sistema Predictivo de la LPF
En este cuaderno construyo las características que utilizaré posteriormente como variables de entrada de los modelos predictivos.
Mi objetivo principal es transformar el historial de partidos de la Liga Panameña de Fútbol en indicadores que representen el estado de cada equipo antes de disputar un encuentro. Para ello calculo variables relacionadas con la forma reciente, puntos obtenidos, producción ofensiva, rendimiento defensivo y comportamiento según la condición de local o visitante.

Durante todo el procedimiento respeto el orden cronológico de los partidos y utilizo únicamente información disponible antes de cada encuentro. De esta manera evito que los resultados del propio partido o información futura sean incorporados dentro de sus características predictivas, reduciendo el riesgo de fuga de información (*data leakage*).

# Celda 2. Carga de librerías y configuración inicial del entorno
En esta celda me encargo de importar las librerías fundamentales para la manipulación de datos y manejo de rutas del sistema de archivos (pandas, numpy y pathlib). Además, configuro las opciones de visualización de pandas para permitir la inspección de conjuntos de datos con un alto volumen de filas y columnas sin que los datos se trunquen en la salida del cuaderno.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# configurar la visualizacion de DataFrames para evitar truncamiento
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


 # Celda 3. Rutas del proyecto y definición del archivo de modelado
Cargo el conjunto histórico previamente depurado y limitado a los partidos considerados elegibles para modelado. A partir de este archivo construiré las características predictivas respetando la secuencia temporal de los encuentros.

En esta celda defino la estructura de directorios del proyecto utilizando la librería pathlib. Establezco la ruta raíz, configuro la carpeta de datos procesados (data/processed) e identifico la ubicación del archivo lpf_historico_modelado.csv, verificando formalmente su existencia en el sistema antes de proceder con el procesamiento.

In [3]:
# definir la ruta raiz del proyecto y el directorio de datos procesados
ROOT = Path("..").resolve()

PROCESSED_DIR = ROOT / "data" / "processed"

ARCHIVO_MODELADO = (
    PROCESSED_DIR
    / "lpf_historico_modelado.csv"
)

# mostrar y verificar las rutas configuradas
print("Raíz del proyecto:")
print(ROOT)

print("\nArchivo utilizado:")
print(ARCHIVO_MODELADO)

print(
    "\nArchivo disponible:",
    ARCHIVO_MODELADO.exists()
)

Raíz del proyecto:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF

Archivo utilizado:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_historico_modelado.csv

Archivo disponible: True


# Celda 4. Verificación del orden temporal estricto
# 1.1. Verificación del orden temporal
Antes de generar variables históricas verifico que los partidos se encuentren ordenados cronológicamente. Esta condición es fundamental, ya que todas las características deben construirse utilizando únicamente encuentros anteriores al partido que intento representar.

En esta celda utilizo la propiedad .is_monotonic_increasing sobre la columna fecha_partido de df_partidos para validar que las observaciones se encuentren dispuestas de manera estrictamente ascendente en el tiempo. Esta verificación asegura que la ingeniería de características posterior respete la secuencia temporal y prevenga la fuga de información hacia el futuro (data leakage).

In [4]:
# cargar el conjunto de datos de modelado con parseo de fechas
df_partidos = pd.read_csv(
    ARCHIVO_MODELADO,
    encoding="utf-8-sig",
    parse_dates=["fecha_partido"]
)

# ordenar cronologicamente por fecha de partido e ID de registro
df_partidos = (
    df_partidos
    .sort_values(
        [
            "fecha_partido",
            "registro_id"
        ]
    )
    .reset_index(drop=True)
)

# verificar dimensiones y cobertura temporal
print(
    "Dimensiones:",
    df_partidos.shape
)

print(
    "Fecha inicial:",
    df_partidos["fecha_partido"].min()
)

print(
    "Fecha final:",
    df_partidos["fecha_partido"].max()
)

print(
    "Partidos:",
    len(df_partidos)
)

# visualizar las primeras 10 filas del dataset ordenado
display(
    df_partidos.head(10)
)

Dimensiones: (2226, 18)
Fecha inicial: 2014-07-18 00:00:00
Fecha final: 2026-09-14 00:00:00
Partidos: 2226


,registro_id,fecha_partido,temporada,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,total_goal_count,resultado,fuente,id_fuente,estado_especial,penales_local,penales_visitante,torneo,tipo_fase,elegible_modelado,archivo_origen
0,LPF_00001,2014-07-18,2014/2015,Atlético Chiriquí,Plaza Amador,0,0,0,E,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
1,LPF_00002,2014-07-18,2014/2015,San Francisco FC,Tauro,0,0,0,E,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
2,LPF_00003,2014-07-18,2014/2015,Árabe Unido,Alianza,0,1,1,V,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
3,LPF_00004,2014-07-19,2014/2015,Chepo,Sporting San Miguelito,1,2,3,V,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
4,LPF_00005,2014-07-19,2014/2015,Chorrillo,CA Independiente de La Chorrera,0,0,0,E,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
5,LPF_00006,2014-07-25,2014/2015,CA Independiente de La Chorrera,Árabe Unido,1,0,1,L,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
6,LPF_00007,2014-07-25,2014/2015,Tauro,Chorrillo,0,1,1,V,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
7,LPF_00008,2014-07-26,2014/2015,Alianza,Chepo,1,0,1,L,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
8,LPF_00009,2014-07-26,2014/2015,Plaza Amador,San Francisco FC,1,2,3,V,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv
9,LPF_00010,2014-07-27,2014/2015,Sporting San Miguelito,Atlético Chiriquí,2,0,2,L,FootyStats,NaN,NaN,NaN,NaN,NaN,NaN,True,panama-lpf-matches-2014-to-2015-stats.csv


# Celda 6. Identificación de variables asociadas al resultado (Control de Data Leakage)
# 1.2. Identificación de variables no disponibles antes del partido
Identifico las variables que describen directamente el resultado del encuentro y que, por lo tanto, no puedo utilizar como características de entrada para predecir ese mismo partido.

Los goles finales y la variable objetivo solamente serán utilizados para calcular el comportamiento histórico de los equipos y para evaluar los modelos. Nunca serán incorporados directamente como predictores del encuentro al que pertenecen.


In [5]:
# definir las columnas que contienen la informacion posterior al partido
columnas_resultado = [
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
]

# listar las variables restringidas para documentar el control de leakage
print(
    "Variables asociadas al resultado del partido:"
)

for columna in columnas_resultado:
    print("-", columna)

Variables asociadas al resultado del partido:
- home_team_goal_count
- away_team_goal_count
- total_goal_count
- resultado


# Celda 7. Transformación a perspectiva individual por equipo (long format)
# 2. Construcción de la perspectiva individual por equipo
Para calcular estadísticas históricas de forma consistente, transformo cada partido en dos observaciones: una desde la perspectiva del equipo local y otra desde la perspectiva del equipo visitante.

Esta estructura me permite reconstruir cronológicamente el historial de cada club y calcular características previas al encuentro, independientemente de la condición en la que haya disputado sus partidos anteriores.

En esta celda construyo las DataFrames df_local y df_visitante reestructurando los atributos de cada partido para reflejar la perspectiva individual del club. Mapeo los resultados globales (L, E, V) a la condición particular del equipo (G, E, P), asigno la condición de localía correspondiente y dejo la estructura lista para la concatenación y cálculo de métricas históricas rodantes.

In [6]:
# perspectiva del equipo local
df_local = pd.DataFrame({
    "registro_id": df_partidos["registro_id"],
    "fecha_partido": df_partidos["fecha_partido"],
    "temporada": df_partidos["temporada"],

    "equipo": df_partidos["home_team_name"],
    "rival": df_partidos["away_team_name"],

    "condicion": "Local",

    "goles_favor": df_partidos["home_team_goal_count"],
    "goles_contra": df_partidos["away_team_goal_count"],

    "resultado_partido": df_partidos["resultado"]
})

df_local["resultado_equipo"] = (
    df_local["resultado_partido"]
    .map({
        "L": "G",
        "E": "E",
        "V": "P"
    })
)


# perspectiva del equipo visitante
df_visitante = pd.DataFrame({
    "registro_id": df_partidos["registro_id"],
    "fecha_partido": df_partidos["fecha_partido"],
    "temporada": df_partidos["temporada"],

    "equipo": df_partidos["away_team_name"],
    "rival": df_partidos["home_team_name"],

    "condicion": "Visitante",

    "goles_favor": df_partidos["away_team_goal_count"],
    "goles_contra": df_partidos["home_team_goal_count"],

    "resultado_partido": df_partidos["resultado"]
})

df_visitante["resultado_equipo"] = (
    df_visitante["resultado_partido"]
    .map({
        "L": "P",
        "E": "E",
        "V": "G"
    })
)

In [7]:
df_historial_equipos = pd.concat(
    [
        df_local,
        df_visitante
    ],
    ignore_index=True
)

df_historial_equipos["puntos"] = (
    df_historial_equipos[
        "resultado_equipo"
    ]
    .map({
        "G": 3,
        "E": 1,
        "P": 0
    })
)

df_historial_equipos[
    "diferencia_goles_partido"
] = (
    df_historial_equipos["goles_favor"]
    - df_historial_equipos["goles_contra"]
)

df_historial_equipos = (
    df_historial_equipos
    .sort_values(
        [
            "equipo",
            "fecha_partido",
            "registro_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Registros por perspectiva de equipo:",
    len(df_historial_equipos)
)

print(
    "Equipos diferentes:",
    df_historial_equipos["equipo"].nunique()
)

Registros por perspectiva de equipo: 4452
Equipos diferentes: 20


# Celda 8. Cálculo de la cantidad de partidos previos por equipo (cumcount)
# 2.1. Cantidad de encuentros previos
Calculo la cantidad de partidos que cada equipo había disputado antes de cada observación. Este indicador me permite identificar encuentros para los cuales existe poco historial previo y será utilizado posteriormente para controlar situaciones de inicio de temporada, nuevos equipos o clubes con participación reciente.

En esta celda utilizo el método .cumcount() agrupado por la columna equipo para calcular el número de partidos disputados con anterioridad a cada fecha. Al iniciar en cero para la primera aparición de un club, el contador representa exactamente la cantidad de registros históricos disponibles en el dataset antes del inicio de dicho encuentro, asegurando un control estricto sobre la madurez del historial de cada equipo.

In [8]:
# calcular la cantidad de partidos previos disputados por cada equipo
df_historial_equipos[
    "partidos_previos"
] = (
    df_historial_equipos
    .groupby("equipo")
    .cumcount()
)

# mostrar las primeras 20 filas con las variables historicas iniciales
display(
    df_historial_equipos[
        [
            "fecha_partido",
            "equipo",
            "rival",
            "partidos_previos",
            "goles_favor",
            "goles_contra",
            "puntos"
        ]
    ].head(20)
)

,fecha_partido,equipo,rival,partidos_previos,goles_favor,goles_contra,puntos
0,2014-07-18,Alianza,Árabe Unido,0,1,0,3
1,2014-07-26,Alianza,Chepo,1,1,0,3
2,2014-08-02,Alianza,Atlético Chiriquí,2,0,1,0
3,2014-08-09,Alianza,San Francisco FC,3,1,3,0
4,2014-08-16,Alianza,Plaza Amador,4,0,0,1
5,2014-08-23,Alianza,Sporting San Miguelito,5,1,4,0
6,2014-08-30,Alianza,Tauro,6,1,2,0
7,2014-09-05,Alianza,CA Independiente de La Chorrera,7,1,0,3
8,2014-09-12,Alianza,Chorrillo,8,1,2,0
9,2014-09-20,Alianza,Árabe Unido,9,3,1,3


# Celda 9. Días transcurridos y tiempo de descanso desde el partido anterior
# 2.2. Días transcurridos desde el partido anterior
Calculo la cantidad de días transcurridos desde el encuentro anterior de cada equipo. Esta variable puede representar diferencias en recuperación, continuidad competitiva o acumulación de partidos.
El cálculo utiliza exclusivamente la fecha del encuentro previo del mismo club.

En esta celda aplico la función .shift(1) agrupada por la columna equipo para obtener la fecha del partido previo (fecha_partido_anterior). A partir de la diferencia temporal entre ambas fechas calculo los días exactos de descanso (dias_descanso), asegurando que el primer partido histórico de cada equipo registre un valor nulo (NaN) por la ausencia de antecedentes en el dataset.

In [9]:
# obtener la fecha del partido previo disputado por el mismo equipo
df_historial_equipos[
    "fecha_partido_anterior"
] = (
    df_historial_equipos
    .groupby("equipo")[
        "fecha_partido"
    ]
    .shift(1)
)

# calcular la cantidad de dias de descanso entre encuentros
df_historial_equipos[
    "dias_descanso"
] = (
    df_historial_equipos[
        "fecha_partido"
    ]
    - df_historial_equipos[
        "fecha_partido_anterior"
    ]
).dt.days

# visualizar el control de descanso y fechas anteriores
display(
    df_historial_equipos[
        [
            "fecha_partido",
            "equipo",
            "fecha_partido_anterior",
            "dias_descanso"
        ]
    ].head(20)
)

,fecha_partido,equipo,fecha_partido_anterior,dias_descanso
0,2014-07-18,Alianza,NaT,NaN
1,2014-07-26,Alianza,2014-07-18,8.0
2,2014-08-02,Alianza,2014-07-26,7.0
3,2014-08-09,Alianza,2014-08-02,7.0
4,2014-08-16,Alianza,2014-08-09,7.0
5,2014-08-23,Alianza,2014-08-16,7.0
6,2014-08-30,Alianza,2014-08-23,7.0
7,2014-09-05,Alianza,2014-08-30,6.0
8,2014-09-12,Alianza,2014-09-05,7.0
9,2014-09-20,Alianza,2014-09-12,8.0


## 3. Construcción de características históricas previas al encuentro

Para generar variables de forma y rendimiento aplico un desplazamiento de una observación antes de calcular cualquier ventana móvil.

De esta manera, los indicadores asociados a un partido solamente contienen información de encuentros anteriores. El partido actual nunca participa en el cálculo de sus propias características.

# Celda 11. Construcción de características de forma reciente (últimos 5 partidos)
# 2.3. Cálculo de métricas móviles recientes
En esta celda implemento la lógica para construir el perfil de forma reciente utilizando una ventana de $5$ partidos 

In [10]:
# definir el tamaño de la ventana para la forma reciente
VENTANA_FORMA = 5

# agrupar el historial por equipo para el cálculo de transformaciones
grupo_equipo = (
    df_historial_equipos
    .groupby("equipo", group_keys=False)
)

# calcular puntos acumulados en los últimos 5 partidos
df_historial_equipos[
    "puntos_ult5"
] = (
    grupo_equipo["puntos"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=VENTANA_FORMA,
            min_periods=1
        )
        .sum()
    )
)

# calcular promedio de puntos por partido (PPG) en los últimos 5 partidos
df_historial_equipos[
    "ppg_ult5"
] = (
    grupo_equipo["puntos"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=VENTANA_FORMA,
            min_periods=1
        )
        .mean()
    )
)

# calcular promedio de goles a favor en los últimos 5 partidos
df_historial_equipos[
    "gf_prom_ult5"
] = (
    grupo_equipo["goles_favor"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=VENTANA_FORMA,
            min_periods=1
        )
        .mean()
    )
)

# calcular promedio de goles en contra en los últimos 5 partidos
df_historial_equipos[
    "gc_prom_ult5"
] = (
    grupo_equipo["goles_contra"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=VENTANA_FORMA,
            min_periods=1
        )
        .mean()
    )
)

# calcular diferencia de goles promedio en los últimos 5 partidos
df_historial_equipos[
    "dg_prom_ult5"
] = (
    grupo_equipo[
        "diferencia_goles_partido"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=VENTANA_FORMA,
            min_periods=1
        )
        .mean()
    )
)

# Celda 12. Verificación visual del desplazamiento shift(1) en un equipo de prueba
# 2.4. Inspección manual del desplazamiento temporal (Audit)
Esta comprobación es fundamental para la auditoría interna y el control de calidad del modelo. Al seleccionar un equipo representativo (en este caso, Tauro) e inspeccionar sus primeros 10 partidos, verifico explícitamente que los promedios y sumas móviles para la fila $N$ se hayan calculado utilizando únicamente los datos observados desde la fila $1$ hasta la fila $N-1$. Esto confirma de manera transparente que no existe fuga de información hacia el presente (target leakage).

In [11]:
# definir equipo de prueba para la verificacion visual de fuga de datos
equipo_prueba = "Tauro"

# seleccionar las columnas de control del historial y promedios moviles
columnas_revision = [
    "fecha_partido",
    "equipo",
    "rival",
    "puntos",
    "puntos_ult5",
    "ppg_ult5",
    "goles_favor",
    "gf_prom_ult5",
    "goles_contra",
    "gc_prom_ult5"
]

# mostrar los primeros 10 registros del equipo de prueba
display(
    df_historial_equipos.loc[
        df_historial_equipos[
            "equipo"
        ].eq(equipo_prueba),
        columnas_revision
    ].head(10)
)

,fecha_partido,equipo,rival,puntos,puntos_ult5,ppg_ult5,goles_favor,gf_prom_ult5,goles_contra,gc_prom_ult5
3061,2014-07-18,Tauro,San Francisco FC,1,NaN,NaN,0,NaN,0,NaN
3062,2014-07-25,Tauro,Chorrillo,0,1.0,1.000000,0,0.00,1,0.000000
3063,2014-08-02,Tauro,Plaza Amador,0,1.0,0.500000,0,0.00,1,0.500000
3064,2014-08-10,Tauro,Árabe Unido,1,1.0,0.333333,1,0.00,1,0.666667
3065,2014-08-17,Tauro,Sporting San Miguelito,0,2.0,0.500000,1,0.25,3,0.750000
3066,2014-08-22,Tauro,Chepo,1,2.0,0.400000,0,0.40,0,1.200000
3067,2014-08-30,Tauro,Alianza,3,2.0,0.400000,2,0.40,1,1.200000
3068,2014-09-06,Tauro,Atlético Chiriquí,1,5.0,1.000000,1,0.80,1,1.200000
3069,2014-09-12,Tauro,CA Independiente de La Chorrera,3,6.0,1.200000,1,1.00,0,1.200000
3070,2014-09-19,Tauro,San Francisco FC,1,8.0,1.600000,1,1.00,1,1.000000


# Celda 13. Auditoría del primer registro por equipo y control de valores nulos iniciales
# 2.5. Verificación de ausencia de antecedentes en la primera observación
En esta celda audito el comportamiento de las características de forma reciente en el primer encuentro histórico de cada equipo. Al utilizar .groupby("equipo").head(1), extraigo la primera aparición de cada club y verifico que todas las variables calculadas con .shift(1) (puntos_ult5, ppg_ult5, gf_prom_ult5, etc.) contengan valores nulos (NaN).

El resultado del conteo debe ser exactamente 0, lo que confirma que ningún equipo hereda información inexistente o futura en su debut dentro del dataset.

In [12]:
# extraer la primera observacion registrada para cada equipo
primeros_partidos = (
    df_historial_equipos
    .groupby("equipo")
    .head(1)
)

# definir la lista de columnas asociadas a la forma reciente
columnas_forma = [
    "puntos_ult5",
    "ppg_ult5",
    "gf_prom_ult5",
    "gc_prom_ult5",
    "dg_prom_ult5"
]

# visualizar el estado inicial de las variables de forma para los debuts de cada club
display(
    primeros_partidos[
        ["equipo"]
        + columnas_forma
    ]
)

# comprobar formalmente que no exista ninguna forma previa cargada en el primer partido
print(
    "Primeros partidos con forma previa disponible:",
    primeros_partidos[
        columnas_forma
    ].notna().any(axis=1).sum()
)

,equipo,puntos_ult5,ppg_ult5,gf_prom_ult5,gc_prom_ult5,dg_prom_ult5
0,Alianza,NaN,NaN,NaN,NaN,NaN
400,Atlético Chiriquí,NaN,NaN,NaN,NaN,NaN
587,Atlético Nacional,NaN,NaN,NaN,NaN,NaN
677,Atlético Veragüense,NaN,NaN,NaN,NaN,NaN
751,CA Independiente de La Chorrera,NaN,NaN,NaN,NaN,NaN
1094,Chepo,NaN,NaN,NaN,NaN,NaN
1168,Chorrillo,NaN,NaN,NaN,NaN,NaN
1325,Costa del Este,NaN,NaN,NaN,NaN,NaN
1515,Costa del Este Academia,NaN,NaN,NaN,NaN,NaN
1531,Herrera,NaN,NaN,NaN,NaN,NaN


Primeros partidos con forma previa disponible: 0


 # Celda 14. Cálculo del rendimiento histórico acumulado previo
# 3.1. Rendimiento histórico previo
Además de la forma reciente, calculo el rendimiento acumulado que cada equipo había registrado antes de disputar cada encuentro.

Para evitar utilizar información del partido actual, calculo primero los valores acumulados y posteriormente excluyo los puntos y goles correspondientes a la observación actual. De esta manera obtengo indicadores históricos de puntos por partido, goles anotados y goles recibidos utilizando exclusivamente encuentros anteriores.

En esta celda obtengo las métricas acumuladas mediante .cumsum(), restando los valores del partido presente para aislar el desempeño previo. Utilizo np.where() condicionado a partidos_previos > 0 para derivar el promedio histórico de puntos (ppg_historico_prev), goles a favor (gf_prom_historico_prev) y goles en contra (gc_prom_historico_prev), garantizando una asignación limpia de valores NaN para el debut de cada club.

In [13]:
# agrupar el historial por equipo manteniendo el orden
grupo_equipo = (
    df_historial_equipos
    .groupby(
        "equipo",
        sort=False
    )
)

# calcular puntos y goles acumulados estrictamente previos al encuentro actual
df_historial_equipos[
    "puntos_acum_prev"
] = (
    grupo_equipo["puntos"].cumsum()
    - df_historial_equipos["puntos"]
)

df_historial_equipos[
    "gf_acum_prev"
] = (
    grupo_equipo["goles_favor"].cumsum()
    - df_historial_equipos["goles_favor"]
)

df_historial_equipos[
    "gc_acum_prev"
] = (
    grupo_equipo["goles_contra"].cumsum()
    - df_historial_equipos["goles_contra"]
)

# calcular el promedio historico acumulado de puntos por partido
df_historial_equipos[
    "ppg_historico_prev"
] = np.where(
    df_historial_equipos["partidos_previos"] > 0,
    (
        df_historial_equipos["puntos_acum_prev"]
        / df_historial_equipos["partidos_previos"]
    ),
    np.nan
)

# calcular el promedio historico acumulado de goles a favor
df_historial_equipos[
    "gf_prom_historico_prev"
] = np.where(
    df_historial_equipos["partidos_previos"] > 0,
    (
        df_historial_equipos["gf_acum_prev"]
        / df_historial_equipos["partidos_previos"]
    ),
    np.nan
)

# calcular el promedio historico acumulado de goles en contra
df_historial_equipos[
    "gc_prom_historico_prev"
] = np.where(
    df_historial_equipos["partidos_previos"] > 0,
    (
        df_historial_equipos["gc_acum_prev"]
        / df_historial_equipos["partidos_previos"]
    ),
    np.nan
)

# visualizar el rendimiento historico acumulado previo de los primeros 20 registros
display(
    df_historial_equipos[
        [
            "fecha_partido",
            "equipo",
            "partidos_previos",
            "ppg_historico_prev",
            "gf_prom_historico_prev",
            "gc_prom_historico_prev"
        ]
    ].head(20)
)

,fecha_partido,equipo,partidos_previos,ppg_historico_prev,gf_prom_historico_prev,gc_prom_historico_prev
0,2014-07-18,Alianza,0,NaN,NaN,NaN
1,2014-07-26,Alianza,1,3.000000,1.000000,0.000000
2,2014-08-02,Alianza,2,3.000000,1.000000,0.000000
3,2014-08-09,Alianza,3,2.000000,0.666667,0.333333
4,2014-08-16,Alianza,4,1.500000,0.750000,1.000000
5,2014-08-23,Alianza,5,1.400000,0.600000,0.800000
6,2014-08-30,Alianza,6,1.166667,0.666667,1.333333
7,2014-09-05,Alianza,7,1.000000,0.714286,1.428571
8,2014-09-12,Alianza,8,1.250000,0.750000,1.250000
9,2014-09-20,Alianza,9,1.111111,0.777778,1.333333


# Celda 15. Rendimiento acumulado dentro de la temporada actual
# 3.2. Rendimiento previo dentro de la temporada
Calculo también indicadores acumulados dentro de cada temporada. Con estas variables represento el desempeño reciente del equipo en el contexto competitivo específico en el que se disputa el partido.

Reinicio los acumulados cuando comienza una nueva temporada, por lo que el primer encuentro de cada club en cada periodo no dispone todavía de estadísticas de temporada.

En esta celda agrupo por la combinación ["equipo", "temporada"] para aislar el rendimiento intrasemanal/intrasemestral de cada equipo. Utilizo .cumcount() para rastrear los partidos jugados previamente dentro del torneo actual (partidos_previos_temporada) y calculo los puntos (puntos_temp_prev), goles a favor (gf_temp_prev) y en contra (gc_temp_prev) acumulados estrictamente antes de la fecha presente. Calculo los promedios de puntos y goles por encuentro en la temporada mediante np.where(), asignando NaN para el debut de cada equipo en cada torneo.

In [14]:
# agrupar el historial por equipo y temporada para aislar el torneo actual
grupo_temporada = (
    df_historial_equipos
    .groupby(
        [
            "equipo",
            "temporada"
        ],
        sort=False
    )
)

# contar la cantidad de partidos previos disputados en la temporada actual
df_historial_equipos[
    "partidos_previos_temporada"
] = (
    grupo_temporada
    .cumcount()
)

# calcular puntos y goles acumulados en la temporada previos al encuentro actual
df_historial_equipos[
    "puntos_temp_prev"
] = (
    grupo_temporada["puntos"].cumsum()
    - df_historial_equipos["puntos"]
)

df_historial_equipos[
    "gf_temp_prev"
] = (
    grupo_temporada["goles_favor"].cumsum()
    - df_historial_equipos["goles_favor"]
)

df_historial_equipos[
    "gc_temp_prev"
] = (
    grupo_temporada["goles_contra"].cumsum()
    - df_historial_equipos["goles_contra"]
)

# calcular promedio de puntos por partido en la temporada actual
df_historial_equipos[
    "ppg_temporada_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_temporada"
    ] > 0,
    (
        df_historial_equipos["puntos_temp_prev"]
        / df_historial_equipos[
            "partidos_previos_temporada"
        ]
    ),
    np.nan
)

# calcular promedio de goles a favor en la temporada actual
df_historial_equipos[
    "gf_prom_temporada_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_temporada"
    ] > 0,
    (
        df_historial_equipos["gf_temp_prev"]
        / df_historial_equipos[
            "partidos_previos_temporada"
        ]
    ),
    np.nan
)

# calcular promedio de goles en contra en la temporada actual
df_historial_equipos[
    "gc_prom_temporada_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_temporada"
    ] > 0,
    (
        df_historial_equipos["gc_temp_prev"]
        / df_historial_equipos[
            "partidos_previos_temporada"
        ]
    ),
    np.nan
)

In [15]:
display(
    df_historial_equipos[
        [
            "fecha_partido",
            "temporada",
            "equipo",
            "partidos_previos_temporada",
            "ppg_temporada_prev",
            "gf_prom_temporada_prev",
            "gc_prom_temporada_prev"
        ]
    ].head(30)
)

,fecha_partido,temporada,equipo,partidos_previos_temporada,ppg_temporada_prev,gf_prom_temporada_prev,gc_prom_temporada_prev
0,2014-07-18,2014/2015,Alianza,0,NaN,NaN,NaN
1,2014-07-26,2014/2015,Alianza,1,3.000000,1.000000,0.000000
2,2014-08-02,2014/2015,Alianza,2,3.000000,1.000000,0.000000
3,2014-08-09,2014/2015,Alianza,3,2.000000,0.666667,0.333333
4,2014-08-16,2014/2015,Alianza,4,1.500000,0.750000,1.000000
5,2014-08-23,2014/2015,Alianza,5,1.400000,0.600000,0.800000
6,2014-08-30,2014/2015,Alianza,6,1.166667,0.666667,1.333333
7,2014-09-05,2014/2015,Alianza,7,1.000000,0.714286,1.428571
8,2014-09-12,2014/2015,Alianza,8,1.250000,0.750000,1.250000
9,2014-09-20,2014/2015,Alianza,9,1.111111,0.777778,1.333333


# Celda 16. Rendimiento histórico acumulado según condición de localía
# 3.3. Rendimiento previo según condición de local o visitante
Calculo el desempeño histórico de cada equipo separando los partidos disputados como local de los jugados como visitante.

Con esta transformación puedo representar, por ejemplo, el rendimiento previo del equipo local únicamente cuando juega en casa y el rendimiento del equipo visitante únicamente cuando juega fuera, evitando mezclar comportamientos que pueden ser diferentes.

En esta celda agrupo por la combinación ["equipo", "condicion"] para aislar el impacto específico del factor localía observado durante la fase del EDA. Utilizo .cumcount() para registrar el número de encuentros disputados previamente en esa misma condición (partidos_previos_condicion) y resto el resultado del partido actual sobre la suma acumulada (.cumsum()). Esto me permite calcular los promedios previos de puntos por partido (ppg_condicion_prev), goles a favor (gf_prom_condicion_prev) y goles en contra (gc_prom_condicion_prev) específicos de la localía o visita antes del comienzo del encuentro.

In [16]:
# agrupar el historial por equipo y condicion (Local/Visitante)
grupo_condicion = (
    df_historial_equipos
    .groupby(
        [
            "equipo",
            "condicion"
        ],
        sort=False
    )
)

# contar la cantidad de partidos previos disputados en la misma condicion
df_historial_equipos[
    "partidos_previos_condicion"
] = (
    grupo_condicion
    .cumcount()
)

# calcular puntos y goles acumulados especificos de la condicion previos al encuentro
df_historial_equipos[
    "puntos_condicion_prev"
] = (
    grupo_condicion["puntos"].cumsum()
    - df_historial_equipos["puntos"]
)

df_historial_equipos[
    "gf_condicion_prev"
] = (
    grupo_condicion["goles_favor"].cumsum()
    - df_historial_equipos["goles_favor"]
)

df_historial_equipos[
    "gc_condicion_prev"
] = (
    grupo_condicion["goles_contra"].cumsum()
    - df_historial_equipos["goles_contra"]
)

# calcular promedio de puntos por partido en la misma condicion
df_historial_equipos[
    "ppg_condicion_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_condicion"
    ] > 0,
    (
        df_historial_equipos[
            "puntos_condicion_prev"
        ]
        / df_historial_equipos[
            "partidos_previos_condicion"
        ]
    ),
    np.nan
)

# calcular promedio de goles a favor en la misma condicion
df_historial_equipos[
    "gf_prom_condicion_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_condicion"
    ] > 0,
    (
        df_historial_equipos[
            "gf_condicion_prev"
        ]
        / df_historial_equipos[
            "partidos_previos_condicion"
        ]
    ),
    np.nan
)

# calcular promedio de goles en contra en la misma condicion
df_historial_equipos[
    "gc_prom_condicion_prev"
] = np.where(
    df_historial_equipos[
        "partidos_previos_condicion"
    ] > 0,
    (
        df_historial_equipos[
            "gc_condicion_prev"
        ]
        / df_historial_equipos[
            "partidos_previos_condicion"
        ]
    ),
    np.nan
)

In [17]:
display(
    df_historial_equipos.loc[
        df_historial_equipos[
            "equipo"
        ].eq("Tauro"),
        [
            "fecha_partido",
            "condicion",
            "partidos_previos_condicion",
            "ppg_condicion_prev",
            "gf_prom_condicion_prev",
            "gc_prom_condicion_prev"
        ]
    ].head(15)
)

,fecha_partido,condicion,partidos_previos_condicion,ppg_condicion_prev,gf_prom_condicion_prev,gc_prom_condicion_prev
3061,2014-07-18,Visitante,0,NaN,NaN,NaN
3062,2014-07-25,Local,0,NaN,NaN,NaN
3063,2014-08-02,Visitante,1,1.000000,0.000000,0.000000
3064,2014-08-10,Local,1,0.000000,0.000000,1.000000
3065,2014-08-17,Visitante,2,0.500000,0.000000,0.500000
3066,2014-08-22,Local,2,0.500000,0.500000,1.000000
3067,2014-08-30,Visitante,3,0.333333,0.333333,1.333333
3068,2014-09-06,Visitante,4,1.000000,0.750000,1.250000
3069,2014-09-12,Local,3,0.666667,0.333333,0.666667
3070,2014-09-19,Local,4,1.250000,0.500000,0.500000


In [18]:
display(
    df_historial_equipos.loc[
        df_historial_equipos[
            "equipo"
        ].eq("Tauro"),
        [
            "fecha_partido",
            "condicion",
            "partidos_previos_condicion",
            "ppg_condicion_prev",
            "gf_prom_condicion_prev",
            "gc_prom_condicion_prev"
        ]
    ].head(15)
)

,fecha_partido,condicion,partidos_previos_condicion,ppg_condicion_prev,gf_prom_condicion_prev,gc_prom_condicion_prev
3061,2014-07-18,Visitante,0,NaN,NaN,NaN
3062,2014-07-25,Local,0,NaN,NaN,NaN
3063,2014-08-02,Visitante,1,1.000000,0.000000,0.000000
3064,2014-08-10,Local,1,0.000000,0.000000,1.000000
3065,2014-08-17,Visitante,2,0.500000,0.000000,0.500000
3066,2014-08-22,Local,2,0.500000,0.500000,1.000000
3067,2014-08-30,Visitante,3,0.333333,0.333333,1.333333
3068,2014-09-06,Visitante,4,1.000000,0.750000,1.250000
3069,2014-09-12,Local,3,0.666667,0.333333,0.666667
3070,2014-09-19,Local,4,1.250000,0.500000,0.500000


# Celda 17. Registro de observaciones efectivas en la ventana de forma reciente
# 3.4. Cantidad de observaciones disponibles para la forma reciente
Como durante los primeros encuentros de un equipo la ventana de cinco partidos todavía no está completa, registro cuántos partidos previos contribuyen realmente al cálculo de la forma.
Esta variable me permite diferenciar, por ejemplo, una media calculada a partir de un único partido de otra calculada utilizando cinco encuentros anteriores.
En esta celda aplico el método .clip(upper=VENTANA_FORMA) sobre la columna partidos_previos para acotar el número de partidos a un máximo de 5. Esta métrica (n_partidos_forma) indica la madurez de la muestra utilizada en los promedios rodantes, permitiendo ponderar o filtrar observaciones según la solidez estadística del promedio calculado.

In [19]:
# acotar el numero de partidos previos aportados a la ventana rodante (maximo 5)
df_historial_equipos[
    "n_partidos_forma"
] = (
    df_historial_equipos[
        "partidos_previos"
    ]
    .clip(
        upper=VENTANA_FORMA
    )
)

# inspeccionar la cantidad de partidos observados vs el promedio móvil calculado
display(
    df_historial_equipos[
        [
            "fecha_partido",
            "equipo",
            "partidos_previos",
            "n_partidos_forma",
            "ppg_ult5"
        ]
    ].head(20)
)

,fecha_partido,equipo,partidos_previos,n_partidos_forma,ppg_ult5
0,2014-07-18,Alianza,0,0,NaN
1,2014-07-26,Alianza,1,1,3.0
2,2014-08-02,Alianza,2,2,3.0
3,2014-08-09,Alianza,3,3,2.0
4,2014-08-16,Alianza,4,4,1.5
5,2014-08-23,Alianza,5,5,1.4
6,2014-08-30,Alianza,6,5,0.8
7,2014-09-05,Alianza,7,5,0.2
8,2014-09-12,Alianza,8,5,0.8
9,2014-09-20,Alianza,9,5,0.8


# Celda 18. Integración y consolidación de características a nivel de partido (Merge)
# 4. Integración de las características por partido
Una vez calculados los indicadores previos de cada equipo, reconstruyo el conjunto a nivel de partido.

Para cada encuentro incorporo las características históricas del equipo local y del visitante en la misma fila. De esta manera obtengo una observación que contiene únicamente información disponible antes del partido junto con las variables objetivo que posteriormente utilizaré durante el entrenamiento y la evaluación.



In [20]:
caracteristicas_equipo = [
    "partidos_previos",
    "dias_descanso",
    "n_partidos_forma",

    "puntos_ult5",
    "ppg_ult5",
    "gf_prom_ult5",
    "gc_prom_ult5",
    "dg_prom_ult5",

    "ppg_historico_prev",
    "gf_prom_historico_prev",
    "gc_prom_historico_prev",

    "partidos_previos_temporada",
    "ppg_temporada_prev",
    "gf_prom_temporada_prev",
    "gc_prom_temporada_prev",

    "partidos_previos_condicion",
    "ppg_condicion_prev",
    "gf_prom_condicion_prev",
    "gc_prom_condicion_prev"
]

# Conjunto Local

In [21]:
features_local = (
    df_historial_equipos.loc[
        df_historial_equipos[
            "condicion"
        ].eq("Local"),
        ["registro_id"]
        + caracteristicas_equipo
    ]
    .copy()
)

features_local = features_local.rename(
    columns={
        columna: f"local_{columna}"
        for columna
        in caracteristicas_equipo
    }
)

# Conjunto Visitante 

In [22]:
features_visitante = (
    df_historial_equipos.loc[
        df_historial_equipos[
            "condicion"
        ].eq("Visitante"),
        ["registro_id"]
        + caracteristicas_equipo
    ]
    .copy()
)

features_visitante = (
    features_visitante.rename(
        columns={
            columna:
            f"visitante_{columna}"
            for columna
            in caracteristicas_equipo
        }
    )
)

# Integracion 

In [23]:
columnas_base = [
    "registro_id",
    "fecha_partido",
    "temporada",
    "home_team_name",
    "away_team_name",

    # objetivos, no predictores directos
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
]

df_features = (
    df_partidos[
        columnas_base
    ]
    .merge(
        features_local,
        on="registro_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        features_visitante,
        on="registro_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "fecha_partido"
    )
    .reset_index(drop=True)
)

print(
    "Dimensiones del dataset:",
    df_features.shape
)

print(
    "Partidos:",
    len(df_features)
)

display(
    df_features.head(10)
)

Dimensiones del dataset: (2226, 47)
Partidos: 2226


,registro_id,fecha_partido,temporada,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,total_goal_count,resultado,local_partidos_previos,local_dias_descanso,local_n_partidos_forma,local_puntos_ult5,local_ppg_ult5,local_gf_prom_ult5,local_gc_prom_ult5,local_dg_prom_ult5,local_ppg_historico_prev,local_gf_prom_historico_prev,local_gc_prom_historico_prev,local_partidos_previos_temporada,local_ppg_temporada_prev,local_gf_prom_temporada_prev,local_gc_prom_temporada_prev,local_partidos_previos_condicion,local_ppg_condicion_prev,local_gf_prom_condicion_prev,local_gc_prom_condicion_prev,visitante_partidos_previos,visitante_dias_descanso,visitante_n_partidos_forma,visitante_puntos_ult5,visitante_ppg_ult5,visitante_gf_prom_ult5,visitante_gc_prom_ult5,visitante_dg_prom_ult5,visitante_ppg_historico_prev,visitante_gf_prom_historico_prev,visitante_gc_prom_historico_prev,visitante_partidos_previos_temporada,visitante_ppg_temporada_prev,visitante_gf_prom_temporada_prev,visitante_gc_prom_temporada_prev,visitante_partidos_previos_condicion,visitante_ppg_condicion_prev,visitante_gf_prom_condicion_prev,visitante_gc_prom_condicion_prev
0,LPF_00001,2014-07-18,2014/2015,Atlético Chiriquí,Plaza Amador,0,0,0,E,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN
1,LPF_00002,2014-07-18,2014/2015,San Francisco FC,Tauro,0,0,0,E,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN
2,LPF_00003,2014-07-18,2014/2015,Árabe Unido,Alianza,0,1,1,V,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN
3,LPF_00004,2014-07-19,2014/2015,Chepo,Sporting San Miguelito,1,2,3,V,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN
4,LPF_00005,2014-07-19,2014/2015,Chorrillo,CA Independiente de La Chorrera,0,0,0,E,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN
5,LPF_00006,2014-07-25,2014/2015,CA Independiente de La Chorrera,Árabe Unido,1,0,1,L,1,6.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN,1,7.0,1,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,1.0,1,0.0,0.0,1.0,0,NaN,NaN,NaN
6,LPF_00007,2014-07-25,2014/2015,Tauro,Chorrillo,0,1,1,V,1,7.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN,1,6.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN
7,LPF_00008,2014-07-26,2014/2015,Alianza,Chepo,1,0,1,L,1,8.0,1,3.0,3.0,1.0,0.0,1.0,3.0,1.0,0.0,1,3.0,1.0,0.0,0,NaN,NaN,NaN,1,7.0,1,0.0,0.0,1.0,2.0,-1.0,0.0,1.0,2.0,1,0.0,1.0,2.0,0,NaN,NaN,NaN
8,LPF_00009,2014-07-26,2014/2015,Plaza Amador,San Francisco FC,1,2,3,V,1,8.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN,1,8.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN
9,LPF_00010,2014-07-27,2014/2015,Sporting San Miguelito,Atlético Chiriquí,2,0,2,L,1,8.0,1,3.0,3.0,2.0,1.0,1.0,3.0,2.0,1.0,1,3.0,2.0,1.0,0,NaN,NaN,NaN,1,9.0,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1.0,0.0,0.0,0,NaN,NaN,NaN


# Celda 18. Integración y consolidación de características a nivel de partido (Merge)
# 4. Integración de las características por partido
Una vez calculados los indicadores previos de cada equipo, reconstruyo el conjunto a nivel de partido.

Para cada encuentro incorporo las características históricas del equipo local y del visitante en la misma fila. De esta manera obtengo una observación que contiene únicamente información disponible antes del partido junto con las variables objetivo que posteriormente utilizaré durante el entrenamiento y la evaluación.

In [24]:
df_features[
    "dif_ppg_ult5"
] = (
    df_features["local_ppg_ult5"]
    - df_features["visitante_ppg_ult5"]
)

df_features[
    "dif_dg_prom_ult5"
] = (
    df_features["local_dg_prom_ult5"]
    - df_features["visitante_dg_prom_ult5"]
)

df_features[
    "dif_ppg_historico"
] = (
    df_features[
        "local_ppg_historico_prev"
    ]
    - df_features[
        "visitante_ppg_historico_prev"
    ]
)

df_features[
    "dif_ppg_temporada"
] = (
    df_features[
        "local_ppg_temporada_prev"
    ]
    - df_features[
        "visitante_ppg_temporada_prev"
    ]
)

df_features[
    "dif_ppg_condicion"
] = (
    df_features[
        "local_ppg_condicion_prev"
    ]
    - df_features[
        "visitante_ppg_condicion_prev"
    ]
)

df_features[
    "dif_dias_descanso"
] = (
    df_features["local_dias_descanso"]
    - df_features[
        "visitante_dias_descanso"
    ]
)

# Demostracion

In [26]:
columnas_diferencia = [
    "registro_id",
    "fecha_partido",
    "home_team_name",
    "away_team_name",

    "dif_ppg_ult5",
    "dif_dg_prom_ult5",
    "dif_ppg_historico",
    "dif_ppg_temporada",
    "dif_ppg_condicion",
    "dif_dias_descanso",

    "resultado"
]

display(
    df_features[
        columnas_diferencia
    ].head(20)
)

,registro_id,fecha_partido,home_team_name,away_team_name,dif_ppg_ult5,dif_dg_prom_ult5,dif_ppg_historico,dif_ppg_temporada,dif_ppg_condicion,dif_dias_descanso,resultado
0,LPF_00001,2014-07-18,Atlético Chiriquí,Plaza Amador,NaN,NaN,NaN,NaN,NaN,NaN,E
1,LPF_00002,2014-07-18,San Francisco FC,Tauro,NaN,NaN,NaN,NaN,NaN,NaN,E
2,LPF_00003,2014-07-18,Árabe Unido,Alianza,NaN,NaN,NaN,NaN,NaN,NaN,V
3,LPF_00004,2014-07-19,Chepo,Sporting San Miguelito,NaN,NaN,NaN,NaN,NaN,NaN,V
4,LPF_00005,2014-07-19,Chorrillo,CA Independiente de La Chorrera,NaN,NaN,NaN,NaN,NaN,NaN,E
5,LPF_00006,2014-07-25,CA Independiente de La Chorrera,Árabe Unido,1.000000,1.000000,1.000000,1.000000,NaN,-1.0,L
6,LPF_00007,2014-07-25,Tauro,Chorrillo,0.000000,0.000000,0.000000,0.000000,NaN,1.0,V
7,LPF_00008,2014-07-26,Alianza,Chepo,3.000000,2.000000,3.000000,3.000000,NaN,1.0,L
8,LPF_00009,2014-07-26,Plaza Amador,San Francisco FC,0.000000,0.000000,0.000000,0.000000,NaN,0.0,V
9,LPF_00010,2014-07-27,Sporting San Miguelito,Atlético Chiriquí,2.000000,1.000000,2.000000,2.000000,NaN,-1.0,L


# Celda 20. Auditoría final de variables y control de fuga de información (Target Leakage)
# 5. Verificación de posibles fugas de información
Antes de exportar el conjunto final verifico que las variables que describen el resultado del partido no se encuentren dentro de la lista de predictores candidatos.Mantengo los goles finales y el resultado únicamente como variables objetivo y de evaluación. No utilizo estas variables como entrada para predecir el mismo encuentro.

En esta celda realizo una validación por intersección de conjuntos (set intersection) entre las variables prohibidas (home_team_goal_count, away_team_goal_count, total_goal_count y resultado) y la lista de predictores identificados por sus prefijos (local_, visitante_, dif_). El conteo de intersecciones prohibidas debe ser estrictamente set() (vacío), confirmando el cumplimiento del protocolo anti-leakage.

In [27]:
# definir el conjunto de columnas asociadas al resultado final del partido
columnas_prohibidas = {
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
}

# filtrar exclusivamente las variables construidas para modelado
columnas_predictoras = (
    [
        columna
        for columna
        in df_features.columns
        if (
            columna.startswith("local_")
            or columna.startswith(
                "visitante_"
            )
            or columna.startswith("dif_")
        )
    ]
)

# evaluar la presencia de variables prohibidas en la lista de predictores
interseccion_prohibida = (
    columnas_prohibidas
    .intersection(
        columnas_predictoras
    )
)

# mostrar el resultado del control de calidad
print(
    "Número de predictores candidatos:",
    len(columnas_predictoras)
)

print(
    "Variables prohibidas encontradas:",
    interseccion_prohibida
)

Número de predictores candidatos: 44
Variables prohibidas encontradas: set()


# Celda 21. Inspección y auditoría de valores faltantes en las características predictivas
# 5.1. Revisión de valores faltantes
Reviso los valores faltantes presentes en las características generadas. Estos valores son esperables principalmente en los primeros encuentros de cada equipo, cuando todavía no existe historial suficiente para calcular determinados indicadores.
En esta etapa no realizo imputación, ya que cualquier procedimiento de sustitución de valores deberá ajustarse exclusivamente con el conjunto de entrenamiento para evitar incorporar información procedente del conjunto de prueba.
En esta celda calculo la cantidad exacta y el porcentaje de valores faltantes (NaN) presentes en las columnas predictoras identificadas. Al no realizar imputaciones prematuras, garantizo la integridad metodológica del proyecto, asegurando que cualquier técnica de imputación (como la mediana o medias móviles) se entrene e impute dentro del pipeline de validación cruzada.

In [28]:
# calcular la cantidad de valores faltantes por columna predictora
faltantes_features = (
    df_features[
        columnas_predictoras
    ]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

# estructurar la informacion en un DataFrame
faltantes_features = pd.DataFrame({
    "faltantes": faltantes_features
})

# calcular el porcentaje de nulos sobre el total de registros
faltantes_features[
    "porcentaje"
] = (
    faltantes_features[
        "faltantes"
    ]
    / len(df_features)
    * 100
).round(2)

# mostrar únicamente las variables que presentan valores nulos
display(
    faltantes_features[
        faltantes_features[
            "faltantes"
        ] > 0
    ]
)

,faltantes,porcentaje
dif_ppg_temporada,73,3.28
local_gc_prom_temporada_prev,73,3.28
local_gf_prom_temporada_prev,73,3.28
local_ppg_temporada_prev,73,3.28
visitante_gc_prom_temporada_prev,72,3.23
visitante_ppg_temporada_prev,72,3.23
visitante_gf_prom_temporada_prev,72,3.23
dif_ppg_condicion,30,1.35
local_ppg_condicion_prev,20,0.90
visitante_gf_prom_condicion_prev,20,0.90


# Celda 22. Validación final de la integridad estructural y temporal del dataset
# 6. Validación final del conjunto de características
Antes de exportar el conjunto de datos verifico su integridad estructural y temporal. Compruebo que cada partido permanezca representado una sola vez, que las variables objetivo no presenten valores faltantes y que no existan valores infinitos generados durante los cálculos.
Estas verificaciones me permiten asegurar que el conjunto resultante puede utilizarse posteriormente para construir los subconjuntos de entrenamiento y prueba.
En esta celda ejecuto las comprobaciones de consistencia clave sobre df_features: verifico la unicidad del identificador de partido (registro_id), garantizo el orden temporal monótono de fecha_partido y corroboro que ninguna de las variables objetivo (resultado, home_team_goal_count, away_team_goal_count) contenga valores nulos que obstaculicen el entrenamiento de los modelos.

In [29]:
# verificar dimensiones y unicidad de registros
print(
    "Número de partidos:",
    len(df_features)
)

print(
    "IDs únicos:",
    df_features["registro_id"].nunique()
)

print(
    "IDs duplicados:",
    df_features["registro_id"].duplicated().sum()
)

# comprobar orden cronológico estricto
print(
    "Fechas ordenadas:",
    df_features["fecha_partido"].is_monotonic_increasing
)

# auditar presencia de nulos en variables objetivo y métricas clave
print(
    "Resultados faltantes:",
    df_features["resultado"].isna().sum()
)

print(
    "Goles locales faltantes:",
    df_features["home_team_goal_count"].isna().sum()
)

print(
    "Goles visitantes faltantes:",
    df_features["away_team_goal_count"].isna().sum()
)

Número de partidos: 2226
IDs únicos: 2226
IDs duplicados: 0
Fechas ordenadas: True
Resultados faltantes: 0
Goles locales faltantes: 0
Goles visitantes faltantes: 0


# Celda 23. Verificación de valores infinitos en las características predictivas
# 6.1. Control de valores no numéricos e infinitos (inf / -inf)
Antes de realizar la exportación del archivo final, realizo un control estricto sobre la presencia de valores infinitos (np.inf o -np.inf) en las columnas numéricas. Estos valores pueden generarse accidentalmente durante divisiones por cero al calcular métricas de promedios o ratios (por ejemplo, al dividir por cero en partidos previos).
En esta celda filtro todas las columnas numéricas dentro del conjunto de predictores candidates (columnas_numericas_predictoras) y aplico np.isinf() para evaluar la matriz de datos completa. Un resultado igual a 0 me confirma la estabilidad matemática de las características construidas.

In [30]:
# identificar las columnas numericas dentro de los predictores
columnas_numericas_predictoras = (
    df_features[
        columnas_predictoras
    ]
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

# evaluar si existen valores infinitos (positivos o negativos) en el conjunto
cantidad_infinitos = (
    np.isinf(
        df_features[
            columnas_numericas_predictoras
        ]
    )
    .sum()
    .sum()
)

# mostrar el resultado del control numérico
print(
    "Valores infinitos en predictores:",
    cantidad_infinitos
)

Valores infinitos en predictores: 0


# Celda 24. Auditoría y catalogación de las características construidas
# 6.1. Resumen de las características construidas
Organizo las variables predictivas en grupos para documentar el tipo de información que representan. Distingo entre experiencia previa, forma reciente, rendimiento histórico, desempeño de temporada, comportamiento según localía y diferencias relativas entre ambos equipos.
En esta celda estructuro un cuadro sintético que agrupa y cuantifica las características generadas en el dataset de modelado. Esta auditoría permite verificar que la dimensión total de predictores coincida con los bloques conceptuales diseñados durante el proceso de ingeniería de características.

In [31]:
# crear el resumen estructurado por grupo de variables predictivas
resumen_features = pd.DataFrame({
    "grupo": [
        "Experiencia previa",
        "Forma reciente",
        "Rendimiento histórico",
        "Rendimiento de temporada",
        "Rendimiento por condición",
        "Diferencias entre equipos"
    ],

    "variables": [
        6,
        12,
        6,
        8,
        8,
        6
    ],

    "descripcion": [
        "Partidos previos, descanso y cantidad de observaciones disponibles.",
        "Puntos, PPG y comportamiento goleador en los últimos cinco partidos.",
        "PPG y promedios históricos de goles previos al encuentro.",
        "Rendimiento acumulado dentro de la temporada antes del partido.",
        "Estadísticas previas diferenciadas como local y visitante.",
        "Comparaciones directas entre las características de ambos equipos."
    ]
})

# mostrar la tabla explicativa del catalogo de predictores
display(resumen_features)

,grupo,variables,descripcion
0,Experiencia previa,6,"Partidos previos, descanso y cantidad de obser..."
1,Forma reciente,12,"Puntos, PPG y comportamiento goleador en los ú..."
2,Rendimiento histórico,6,PPG y promedios históricos de goles previos al...
3,Rendimiento de temporada,8,Rendimiento acumulado dentro de la temporada a...
4,Rendimiento por condición,8,Estadísticas previas diferenciadas como local ...
5,Diferencias entre equipos,6,Comparaciones directas entre las característic...


# Celda 25. Extracción e integración de variables de calendario prepartido
# 6.2. Variables temporales del encuentro
Añado información básica del calendario conocida antes de disputarse el partido. Utilizo el mes y el día de la semana como posibles variables auxiliares para representar variaciones asociadas al calendario competitivo.
No utilizo directamente el marcador ni ninguna información generada durante o después del encuentro.
En esta celda extraigo el mes (mes_partido) y el día de la semana (dia_semana) a partir de la columna de fecha (fecha_partido) con los descriptores .dt.month y .dt.dayofweek. Posteriormente, incorporo estas dos variables al listado oficial de columnas_predictoras para ampliar el contexto temporal antes del modelado.

In [32]:
# extraer el mes del encuentro (1-12)
df_features[
    "mes_partido"
] = (
    df_features[
        "fecha_partido"
    ].dt.month
)

# extraer el dia de la semana del encuentro (0=Lunes, 6=Domingo)
df_features[
    "dia_semana"
] = (
    df_features[
        "fecha_partido"
    ].dt.dayofweek
)

# incorporar las variables temporales a la lista de predictores candidatos
columnas_predictoras.extend([
    "mes_partido",
    "dia_semana"
])

# verificar el recuento actualizado de predictores
print(
    "Predictores numéricos candidatos:",
    len(columnas_predictoras)
)

Predictores numéricos candidatos: 46


# Celda 26. Identificación y conservación de variables categóricas de equipo
# 6.3. Identidad de los equipos
Conservo los nombres del equipo local y visitante como variables categóricas candidatas. No asumo que deban formar parte obligatoriamente de todos los modelos.Compararé el rendimiento de configuraciones con y sin la identidad directa de los clubes, ya que mi objetivo principal es que los modelos aprendan patrones deportivos mediante características dinámicas y no dependan exclusivamente del nombre de cada equipo.

Los clubes históricos permanecen dentro del conjunto de entrenamiento debido al valor de sus partidos anteriores. La interfaz de predicción futura utilizará de manera separada el listado de clubes presentes en el periodo competitivo más reciente.

En esta celda defino formalmente la lista de variables categóricas candidatas (home_team_name y away_team_name). Al mantenerlas explícitas, garantizo la flexibilidad para experimentar con algoritmos que manejan atributos categóricos de forma nativa (como LightGBM o CatBoost) frente a modelos basados en vectores exclusivamente numéricos.

In [33]:
# definir las columnas categoricas que identifican a los equipos
columnas_categoricas = [
    "home_team_name",
    "away_team_name"
]

# mostrar el listado de variables categoricas candidatas
print(
    "Variables categóricas candidatas:"
)

for columna in columnas_categoricas:
    print("-", columna)

Variables categóricas candidatas:
- home_team_name
- away_team_name


# Celda 27. Consolidación de la lista definitiva de variables candidatas para el modelado
# 6.4. Consolidación del espacio de características (Feature Space)
Sintetizo el listado final de variables candidatas combinando los predictores numéricos derivados (experiencia, forma reciente, promedios históricos, rendimiento de temporada, características por condición de localía, diferenciales y variables de calendario) junto con las variables categóricas de identidad de los equipos.

In [35]:
# concatenar predictores numericos y categoricos candidatas
columnas_modelado = (
    columnas_predictoras
    + columnas_categoricas
)

# mostrar el recuento total y desglosado de variables candidatas
print(
    "Predictores numéricos:",
    len(columnas_predictoras)
)

print(
    "Predictores categóricos:",
    len(columnas_categoricas)
)

print(
    "Total de predictores candidatos:",
    len(columnas_modelado)
)

Predictores numéricos: 46
Predictores categóricos: 2
Total de predictores candidatos: 48


# Celda 28. Último control estricto de fuga de información 
# 6.5. Verificación automatizada mediante Assert
Como última medida de seguridad antes de proceder a la exportación final del conjunto de datos, ejecuto una prueba de aserción (assert) sobre la lista consolidada columnas_modelado.

In [36]:
# definir el conjunto estricto de variables objetivo prohibidas como predictores
variables_no_permitidas = {
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
}

# calcular la interseccion entre las variables prohibidas y las columnas de modelado
fuga_detectada = (
    variables_no_permitidas
    .intersection(
        set(columnas_modelado)
    )
)

# mostrar el resultado del control
print(
    "Variables objetivo presentes como predictores:",
    fuga_detectada
)

# validar formalmente que la interseccion sea completamente vacia
assert len(fuga_detectada) == 0

Variables objetivo presentes como predictores: set()


# Celda 29. Construcción del conjunto final de características para exportación
# 7. Construcción del conjunto final de características
Construyo el archivo final manteniendo los identificadores del partido, la información temporal, los equipos participantes, las características predictivas generadas y las variables objetivo. Los goles finales y el resultado permanecen dentro del archivo únicamente para facilitar posteriormente el entrenamiento y la evaluación. Estas columnas no serán incorporadas al conjunto de variables de entrada (X).

In [38]:
# definir los grupos de columnas para el esquema de exportacion
columnas_identificacion = [
    "registro_id",
    "fecha_partido",
    "temporada"
]

columnas_objetivo = [
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count",
    "resultado"
]

# estructurar el orden final de todas las columnas del dataset
columnas_exportar = (
    columnas_identificacion
    + columnas_categoricas
    + columnas_predictoras
    + columnas_objetivo
)

# construir y ordenar el DataFrame final de modelado
df_features_final = (
    df_features[
        columnas_exportar
    ]
    .copy()
    .sort_values(
        [
            "fecha_partido",
            "registro_id"
        ]
    )
    .reset_index(drop=True)
)

# mostrar dimensiones finales y primeras observaciones
print(
    "Dimensiones finales:",
    df_features_final.shape
)

display(
    df_features_final.head()
)

Dimensiones finales: (2226, 55)


,registro_id,fecha_partido,temporada,home_team_name,away_team_name,local_partidos_previos,local_dias_descanso,local_n_partidos_forma,local_puntos_ult5,local_ppg_ult5,local_gf_prom_ult5,local_gc_prom_ult5,local_dg_prom_ult5,local_ppg_historico_prev,local_gf_prom_historico_prev,local_gc_prom_historico_prev,local_partidos_previos_temporada,local_ppg_temporada_prev,local_gf_prom_temporada_prev,local_gc_prom_temporada_prev,local_partidos_previos_condicion,local_ppg_condicion_prev,local_gf_prom_condicion_prev,local_gc_prom_condicion_prev,visitante_partidos_previos,visitante_dias_descanso,visitante_n_partidos_forma,visitante_puntos_ult5,visitante_ppg_ult5,visitante_gf_prom_ult5,visitante_gc_prom_ult5,visitante_dg_prom_ult5,visitante_ppg_historico_prev,visitante_gf_prom_historico_prev,visitante_gc_prom_historico_prev,visitante_partidos_previos_temporada,visitante_ppg_temporada_prev,visitante_gf_prom_temporada_prev,visitante_gc_prom_temporada_prev,visitante_partidos_previos_condicion,visitante_ppg_condicion_prev,visitante_gf_prom_condicion_prev,visitante_gc_prom_condicion_prev,dif_ppg_ult5,dif_dg_prom_ult5,dif_ppg_historico,dif_ppg_temporada,dif_ppg_condicion,dif_dias_descanso,mes_partido,dia_semana,home_team_goal_count,away_team_goal_count,total_goal_count,resultado
0,LPF_00001,2014-07-18,2014/2015,Atlético Chiriquí,Plaza Amador,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,4,0,0,0,E
1,LPF_00002,2014-07-18,2014/2015,San Francisco FC,Tauro,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,4,0,0,0,E
2,LPF_00003,2014-07-18,2014/2015,Árabe Unido,Alianza,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,4,0,1,1,V
3,LPF_00004,2014-07-19,2014/2015,Chepo,Sporting San Miguelito,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,5,1,2,3,V
4,LPF_00005,2014-07-19,2014/2015,Chorrillo,CA Independiente de La Chorrera,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,5,0,0,0,E


# Celda 30. Exportación final del conjunto de características procesado
# 8. Exportación del conjunto de características
Exporto el conjunto generado para utilizarlo en las siguientes etapas del sistema predictivo. Mantengo una fila por partido y conservo el orden cronológico. La imputación de valores faltantes, codificación de variables categóricas y escalado de variables numéricas no se realizan en este cuaderno. Estos procedimientos serán ajustados posteriormente utilizando exclusivamente el conjunto de entrenamiento.

En esta celda guardo el DataFrame df_features_final en formato CSV dentro del directorio de datos procesados (PROCESSED_DIR). Utilizo index=False para no arrastrar índices numéricos adicionales y la codificación utf-8-sig para asegurar compatibilidad de caracteres especiales. Valido formalmente la existencia física del archivo generado y sus dimensiones finales.

In [39]:
# definir la ruta de destino para el archivo procesado de modelado
ARCHIVO_FEATURES = (
    PROCESSED_DIR
    / "lpf_features_modelado.csv"
)

# exportar la matriz final a formato CSV
df_features_final.to_csv(
    ARCHIVO_FEATURES,
    index=False,
    encoding="utf-8-sig"
)

# mostrar reporte de confirmación de exportación
print(
    "Archivo exportado:"
)

print(
    ARCHIVO_FEATURES
)

print(
    "\nExiste:",
    ARCHIVO_FEATURES.exists()
)

print(
    "Filas:",
    len(df_features_final)
)

print(
    "Columnas:",
    df_features_final.shape[1]
)

Archivo exportado:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_features_modelado.csv

Existe: True
Filas: 2226
Columnas: 55


# Celda 31. Comprobación de lectura del archivo exportado
# 8.1. Verificación de integridad post-exportación
Como último paso de control de calidad dentro del flujo de trabajo de este cuaderno, reimporto el archivo lpf_features_modelado.csv recién generado para confirmar que el proceso de escritura en disco fue exitoso y no corrupto.
En esta celda leo el CSV indicando el argumento parse_dates=["fecha_partido"] para reconvertir la columna de fechas a objetos datetime. Evalúo que las dimensiones recuperadas coincidan exactamente con la matriz exportada, confirmo que la unicidad de los identificadores (registro_id) se mantenga intacta, verifico el rango temporal completo (fecha inicial y final) y aseguro que no existan valores nulos en la variable objetivo (resultado).

In [40]:
# reimportar el archivo exportado para comprobar la integridad de la lectura
df_verificacion = pd.read_csv(
    ARCHIVO_FEATURES,
    encoding="utf-8-sig",
    parse_dates=["fecha_partido"]
)

# mostrar métricas de verificación
print(
    "Dimensiones recuperadas:",
    df_verificacion.shape
)

print(
    "IDs únicos:",
    df_verificacion[
        "registro_id"
    ].nunique()
)

print(
    "Fecha inicial:",
    df_verificacion[
        "fecha_partido"
    ].min()
)

print(
    "Fecha final:",
    df_verificacion[
        "fecha_partido"
    ].max()
)

print(
    "Resultado completo:",
    df_verificacion[
        "resultado"
    ].notna().all()
)

Dimensiones recuperadas: (2226, 55)
IDs únicos: 2226
Fecha inicial: 2014-07-18 00:00:00
Fecha final: 2026-09-14 00:00:00
Resultado completo: True
